In [1]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import pandas as pd
import os
import gc
import zipfile
from numba import njit
from tqdm import tqdm

# ==========================================
# 1. ネットワーク生成 (Numba JIT)
# ==========================================
@njit
def build_ba_edges_fast(N, m):
    num_edges = 2 * m * N
    row = np.empty(num_edges, dtype=np.int32)
    col = np.empty(num_edges, dtype=np.int32)
    targets = np.empty(num_edges, dtype=np.int32)
    idx = 0; target_idx = 0

    for i in range(m):
        for j in range(m):
            if i != j:
                row[idx] = i; col[idx] = j; idx += 1
                targets[target_idx] = i; target_idx += 1

    chosen = np.empty(m, dtype=np.int32)
    for i in range(m, N):
        c_idx = 0
        while c_idx < m:
            r = np.random.randint(0, target_idx)
            node = targets[r]
            already = False
            for k in range(c_idx):
                if chosen[k] == node: already = True; break
            if not already:
                chosen[c_idx] = node; c_idx += 1

        for k in range(m):
            target = chosen[k]
            row[idx] = i; col[idx] = target; idx += 1
            row[idx] = target; col[idx] = i; idx += 1
            targets[target_idx] = i; target_idx += 1
            targets[target_idx] = target; target_idx += 1

    return row[:idx], col[:idx]

def build_clustered_network(N, m, weak_ties_ratio=0.002):
    N_A = N // 2
    N_B = N - N_A
    row_A, col_A = build_ba_edges_fast(N_A, m)
    row_B, col_B = build_ba_edges_fast(N_B, m)
    row_B = row_B + N_A; col_B = col_B + N_A

    row = np.concatenate([row_A, row_B])
    col = np.concatenate([col_A, col_B])

    num_weak_ties = max(1, int(N * weak_ties_ratio))
    wt_row = np.random.randint(0, N_A, num_weak_ties)
    wt_col = np.random.randint(N_A, N, num_weak_ties)

    row = np.concatenate([row, wt_row, wt_col])
    col = np.concatenate([col, wt_col, wt_row])

    data = np.ones(len(row), dtype=np.float32)
    A_sparse = sp.csr_matrix((data, (row, col)), shape=(N, N))

    deg_A = np.bincount(row_A, minlength=N_A)
    hub_A = np.argmax(deg_A)
    boundary_nodes = np.unique(np.concatenate([wt_row, wt_col]))

    return A_sparse, N_A, hub_A, boundary_nodes

# ==========================================
# 2. 敵対的ゲーム・シミュレーション
# ==========================================
def run_adversarial_simulation(N, A_sparse, N_A, hub_A, boundary_nodes, params, scenario):
    dt = params['dt']
    steps = params['steps']
    theta_detect = params['theta_detect']

    degrees = np.array(A_sparse.sum(axis=1)).flatten()
    degrees[degrees == 0] = 1.0

    # 初期値依存を完全に排除（一様乱数）
    np.random.seed()
    X = np.random.uniform(0, 0.2, N).astype(np.float32)
    D = np.random.uniform(0, 0.1, N).astype(np.float32)

    v_target = np.full(N, 0.3, dtype=np.float32)
    v_target[:N_A] = 1.0

    triad_boost = np.ones(N, dtype=np.float32)
    b = np.zeros(N, dtype=np.float32)

    history_D_B = []
    history_X_A = []

    intervened = False
    intervention_time = -1

    for step in range(steps):
        current_X_A = np.mean(X[:N_A])
        history_X_A.append(current_X_A)

        # --------------------------------------------------
        # [攻撃者側 (Hub)] の不完全情報ゲーム戦略
        # --------------------------------------------------
        if scenario == 'Naive':
            # 愚直な攻撃：何も考えずに強烈なフェイクを流す
            b[hub_A] = 0.95 if step < 50 else 0.0

        elif scenario == 'Stealth':
            # ステルス攻撃：プラットフォームの検知閾値(theta)を下回るように動的制御
            if current_X_A < (theta_detect - 0.005):
                b[hub_A] = 0.8  # 閾値まで余裕があれば引き上げる
            else:
                b[hub_A] = 0.05 # 閾値に近づいたら潜伏する

        # --------------------------------------------------
        # [プラットフォーム側 (Algorithm)] の動的検知・予防介入
        # --------------------------------------------------
        if not intervened and current_X_A > theta_detect:
            intervened = True
            intervention_time = step
            # 検知と同時に2-単体（トライアド）を境界に構築
            triad_boost[boundary_nodes] = params['triad_multiplier']

        # SDE更新
        P = A_sparse.dot(X) / degrees
        noise = np.random.normal(0, 1, N).astype(np.float32) * np.sqrt(dt)
        X_new = np.clip(X + (P + b - X) * dt + params['sigma'] * noise, 0, 1)

        repair_capacity = np.exp(-params['gamma'] * D)
        dD = (params['k_break'] * v_target * X_new * (1 - D) -
              (params['k_restore'] * triad_boost) * (1 - X_new) * D * repair_capacity) * dt
        D_new = np.clip(D + dD, 0, 1)

        X = X_new
        D = D_new

        history_D_B.append(np.mean(D[N_A:]))

    return history_D_B, history_X_A, intervention_time

# ==========================================
# 3. メイン実行スクリプト
# ==========================================
def main():
    N_list = [1000, 10000, 100000, 1000000, 10000000]
    m = 2

    params = {
        'dt': 0.1,
        'steps': 200,          # ステルスの長期効果を見るため延長
        'theta_detect': 0.18,  # プラットフォームの異常検知閾値
        'sigma': 0.05,
        'k_break': 0.6,
        'k_restore': 0.1,
        'gamma': 6.0,
        'triad_multiplier': 5.0
    }

    output_dir = "tmfg_adversarial_results"
    os.makedirs(output_dir, exist_ok=True)
    summary_data = []

    for N in N_list:
        print(f"\n[{N} Nodes] ネットワーク生成中...")
        try:
            A_sparse, N_A, hub_A, boundary_nodes = build_clustered_network(N, m)
            print(f"-> 敵対的ゲーム・シミュレーション実行中...")

            # Naive Attack (愚直な攻撃 vs 動的検知)
            D_B_naive, X_A_naive, t_int_naive = run_adversarial_simulation(
                N, A_sparse, N_A, hub_A, boundary_nodes, params, 'Naive'
            )

            # Stealth Attack (ステルス攻撃 vs 動的検知)
            D_B_stealth, X_A_stealth, t_int_stealth = run_adversarial_simulation(
                N, A_sparse, N_A, hub_A, boundary_nodes, params, 'Stealth'
            )

            del A_sparse
            gc.collect()

            # CSV出力
            df = pd.DataFrame({
                'TimeStep': np.arange(params['steps']),
                'D_B_Naive': D_B_naive,
                'D_B_Stealth': D_B_stealth,
                'X_A_Naive': X_A_naive,
                'X_A_Stealth': X_A_stealth
            })
            df.to_csv(os.path.join(output_dir, f'adversarial_N{N}.csv'), index=False)

            # グラフ1: プラットフォームの検知変数 (X_A) の推移と閾値の攻防
            plt.figure(figsize=(10, 5))
            plt.plot(df['TimeStep'], df['X_A_Naive'], 'r--', linewidth=2, label='Naive Attack (Exposed)')
            plt.plot(df['TimeStep'], df['X_A_Stealth'], 'b-', linewidth=2, label='Stealth Attack (Evasive)')
            plt.axhline(y=params['theta_detect'], color='k', linestyle=':', label='Detection Threshold $\\theta$')
            if t_int_naive > 0:
                plt.axvline(x=t_int_naive, color='r', linestyle=':', label=f'Naive Intervened at t={t_int_naive}')
            plt.title(f'Adversarial Dynamics: Evading Detection Threshold (N={N:,})')
            plt.ylabel('Conformity $X_A$ (Sensor Variable)')
            plt.legend(); plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'plot_sensor_N{N}.png'), dpi=300, bbox_inches='tight')
            plt.close()

            # グラフ2: マジョリティへのダメージカスケード (D_B)
            plt.figure(figsize=(10, 5))
            plt.plot(df['TimeStep'], df['D_B_Naive'], 'r--', linewidth=2, label='Systemic Damage (Naive)')
            plt.plot(df['TimeStep'], df['D_B_Stealth'], 'b-', linewidth=3, label='Systemic Damage (Stealth)')
            plt.title(f'Systemic Metastasis under Adversarial Attacks (N={N:,})')
            plt.ylabel('Majority Defect $D_B$')
            plt.legend(); plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'plot_damage_N{N}.png'), dpi=300, bbox_inches='tight')
            plt.close()

            summary_data.append({
                'N': N,
                'Int_Time_Naive': t_int_naive,
                'Int_Time_Stealth': t_int_stealth,
                'Final_DB_Naive': D_B_naive[-1],
                'Final_DB_Stealth': D_B_stealth[-1]
            })

        except MemoryError:
            print(f"[警告] N={N} でメモリ不足。")
            break

    # サマリーとZIP化
    df_sum = pd.DataFrame(summary_data)
    df_sum.to_csv(os.path.join(output_dir, 'adversarial_summary.csv'), index=False)

    zip_filename = 'TMFG_Adversarial_Results.zip'
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(output_dir):
            for file in files:
                zipf.write(os.path.join(root, file), arcname=file)

    print(f"\n🎉 完了: 結果を '{zip_filename}' に出力しました。")
    print(df_sum.to_string())

if __name__ == "__main__":
    main()


[1000 Nodes] ネットワーク生成中...
-> 敵対的ゲーム・シミュレーション実行中...

[10000 Nodes] ネットワーク生成中...
-> 敵対的ゲーム・シミュレーション実行中...

[100000 Nodes] ネットワーク生成中...
-> 敵対的ゲーム・シミュレーション実行中...

[1000000 Nodes] ネットワーク生成中...
-> 敵対的ゲーム・シミュレーション実行中...

[10000000 Nodes] ネットワーク生成中...
-> 敵対的ゲーム・シミュレーション実行中...

🎉 完了: 結果を 'TMFG_Adversarial_Results.zip' に出力しました。
          N  Int_Time_Naive  Int_Time_Stealth  Final_DB_Naive  Final_DB_Stealth
0      1000              50                43        0.243204          0.257350
1     10000              -1                -1        0.273180          0.261614
2    100000              -1                -1        0.266008          0.264494
3   1000000              -1                -1        0.264914          0.263879
4  10000000              -1                -1        0.264720          0.263514
